In [ ]:
import os
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

os.environ["LANGCHAIN_TRACING_V2"] = "false"
from dotenv import load_dotenv
import getpass
load_dotenv(".env.RAG")
hf_token=os.getenv("HF_TOKEN")

from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi

from langchain_huggingface import HuggingFaceEndpoint, HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from transformers import pipeline

from sentence_transformers import CrossEncoder
import re

import gradio as gr
import time



reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")



In [5]:
folder_path="./Data"

loader = DirectoryLoader(
     folder_path,
     glob="**/*",


)

In [ ]:



docs = loader.load()
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)


def cleaner_func(docs):

  if isinstance(docs, list):
    cleared=[]
    for doc in docs:
      if isinstance(doc, str):
         doc = re.sub(r'\n+', '\n', doc)
         doc = re.sub(r'\s+', ' ', doc)
         cleared.append(doc.strip())
      else:
        cleared.append(doc)
    return cleared

    

  elif isinstance(docs, str):
    doc= ''.join(str(d)for d in docs)

  docs = re.sub(r'\n+', '\n', docs)
  docs = re.sub(r'\s+', ' ', docs)
  return docs.strip()

cleaned_docs = cleaner_func(docs)
pipline= pipeline(
    "text-generation",
    model="microsoft/phi-2",
    max_new_tokens=512,
    temperature=0.3
)
llm = HuggingFacePipeline(pipeline=pipline) 
template = """Given the following context, answer the question accurately.
Context: {context}
Question: {question}
Answer:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])
qa_chain = LLMChain(llm=llm, prompt=prompt)

 

 

text_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,  
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
    length_function=len,
)
cleaned_chunk = text_split.split_documents(cleaned_docs)


vector_store = Chroma.add_documents(
    documents=cleaned_chunk,
    collection_name="Lang_Rag_hf", 
    embedding=embd,
    persist_directory="./Rag/chroma_DB",
    
)
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":7}
)
 

text_chunks = [chunk.page_content for chunk in cleaned_chunk]
tokenized_chunks = [chunk.split() for chunk in text_chunks]
bm25 = BM25Okapi(tokenized_chunks)


 

def retrieve(query, top_k=5):
    

    vector_docs = retriever.invoke(query)
    vector_texts = [doc.page_content for doc in vector_docs]
    
    tokenized_query = query.split()
    bm25_texts = bm25.get_top_n(tokenized_query, text_chunks, n=top_k)
    
    all_texts = []
    seen = set()
    for text in bm25_texts + vector_texts:
        if text not in seen:
            seen.add(text)
            all_texts.append(text)
    
    return all_texts[:top_k*2]

def rerank(query, texts):
    if 'reranker' not in globals():
        return texts[:3]
    
    pairs = [[query, text] for text in texts]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, texts), reverse=True)
    return [text for score, text in ranked[:3]]

def ask(question):
    
    texts = retrieve(question)
    
    top_texts = rerank(question, texts)
    
    if 'qa_chain' in globals():
        context = "\n\n".join(top_texts)
        response = qa_chain.run(context=context, question=question)
        if "Answer:" in response:
            response = response.split("Answer:")[-1].strip()
        return response
    else:
        return f"Found {len(top_texts)} relevant passages."

# print(ask("Which university does Hunter Jacobson studies mentioned in the resume? He is a student of human resource"))




print("Testing ask function:")
test_result = ask("test")
print(f"ask test result: {type(test_result)}")

def gradio_chat(message, history):
    """Simple wrapper for ask function"""
    try:
        
        result = ask(message)
        
        
        if not isinstance(result, str):
            result = str(result)
        
        return result
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(error_msg)
        return error_msg

#
demo = gr.ChatInterface(
    fn=gradio_chat,
    title="RAG Document Chatbot",
    description="Ask questions",
    examples=[
        "Which university does Hunter Jacobson studies mentioned in the resume? He is a student of human resource"
    ]
)



demo.launch(share=False, server_port=7861, debug=True)

Hunter Jacobson Human Resources Manager Human Resources student looking to leverage my experience in recruiting for technical roles to help a mission-driven company like Panorama grow their team when 

{'source': 'Data\\res10.pdf'}


Loading weights: 100%|██████████| 453/453 [00:00<00:00, 3687.13it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
C:\Users\ghali\AppData\Local\Temp\ipykernel_13032\1946925668.py:41: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=prompt)


TypeError: Chroma.__init__() got an unexpected keyword argument 'documents'